# TEEN (NeurIPS 2023) — Clean-Room Reproduction Pipeline (CIFAR-100)

Notebook điều phối toàn bộ quy trình tái lập thực nghiệm độc lập (*clean-room implementation*) cho bài báo:
> **Few-Shot Class-Incremental Learning via Training-Free Prototype Calibration** (Wang et al., NeurIPS 2023)

- **Nhóm thực hiện:** Nhóm 05 — Nguyễn Hoàng Long & Trần Gia Hiên
- **Dự án Weights & Biases công khai:** [https://wandb.ai/tghien/Intro2ML-Group05](https://wandb.ai/tghien/Intro2ML-Group05)
- **Phạm vi thực nghiệm (Core Scope):**
  1. Huấn luyện mạng **ResNet-20** ($d = 64$) + **Cosine Classifier** ($\tau = 16.0$) trên 60 lớp cơ sở của CIFAR-100 (`Session 0`).
  2. Đánh giá tăng dần không huấn luyện lại (*training-free calibration*, $\alpha = 0.1, \tau = 16.0$) qua 8 phiên **5-way 5-shot** (`Session 1..8`).
  3. Khảo sát độ nhạy siêu tham số trên 40 tổ hợp $\alpha \in [0.0, 0.9]$ và $\tau \in \{8, 16, 32, 64\}$ (Section 6.3, Figure 5).

In [ ]:
import os
import torch

print("Thiết bị tính toán:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (Khuyến nghị bật T4 GPU)")

# Kéo mã nguồn từ GitHub nếu chạy trực tiếp từ link Colab
REPO_DIR = "/content/Intro2ML_Adopt-ML-paper"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/famevease/Intro2ML_Adopt-ML-paper.git {REPO_DIR}


In [ ]:
%cd {REPO_DIR}
!git pull origin main
%pip install -q -r requirements.txt

In [ ]:
import os
import wandb

try:
    from google.colab import drive  # type: ignore

    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/IntroML_TEEN/checkpoints"
except ImportError:
    CKPT_DIR = "./checkpoints"

DATA_DIR = "/content/data" if os.path.exists("/content") else "./data"
CKPT_PATH = os.path.join(CKPT_DIR, "base_model.pth")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Thư mục dữ liệu:", DATA_DIR)
print("Đường dẫn checkpoint:", CKPT_PATH)

wandb.login()

In [ ]:
FORCE_RETRAIN = False  # Đổi thành True nếu muốn train lại 100 epochs (~15-20 phút trên T4)

if FORCE_RETRAIN or not os.path.exists(CKPT_PATH):
    !python -m src.train_base \
        --data_dir {DATA_DIR} \
        --save_dir {CKPT_DIR} \
        --epochs 100 \
        --batch_size 256 \
        --lr 0.1 \
        --tau 16.0 \
        --seed 2026
else:
    print(f"Đã tìm thấy checkpoint Session 0 tại: {CKPT_PATH} (Bỏ qua bước train lại).")

In [ ]:
# Chạy đánh giá tăng dần không huấn luyện lại (alpha=0.1, tau=16.0, seed=2026)
!python -m src.eval_incremental \
    --data_dir {DATA_DIR} \
    --ckpt_path {CKPT_PATH} \
    --alpha 0.1 \
    --tau 16.0 \
    --seed 2026

In [ ]:
# Quét lưới alpha in [0.0..0.9] và tau in {8, 16, 32, 64} theo Section 6.3 (Figure 5)
!python -m src.sweep_alpha \
    --data_dir {DATA_DIR} \
    --ckpt_path {CKPT_PATH} \
    --seed 2026


In [ ]:
import matplotlib.pyplot as plt

sessions = list(range(9))
paper_acc = [74.92, 72.65, 68.74, 65.01, 62.01, 59.29, 57.90, 54.76, 52.64]
rebuild_avg = [70.67, 66.15, 62.09, 58.00, 54.60, 51.89, 49.66, 47.28, 45.07]
rebuild_base = [70.67, 69.92, 69.48, 69.15, 68.83, 68.35, 68.02, 67.82, 67.15]

plt.figure(figsize=(8, 4.5), dpi=150)
plt.plot(sessions, paper_acc, "-s", linewidth=2, label="TEEN Paper (Table 6, RTX 3090)")
plt.plot(sessions, rebuild_avg, "-o", linewidth=2, label="Clean-Room Rebuild AvgAcc (Colab T4)")
plt.plot(sessions, rebuild_base, "--^", linewidth=1.8, label="Clean-Room Rebuild BAcc (60 Base Classes)")

plt.title("CIFAR-100 FSCIL Reproduction: Session 0 to Session 8", fontweight="bold")
plt.xlabel("Incremental Session")
plt.ylabel("Top-1 Accuracy (%)")
plt.xticks(sessions)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()